In [1]:
import socket
import time
from io import StringIO
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

import pandas as pd

# -----------------------------
# 1. Choose your seasons
# -----------------------------
START_YEAR = 2023
END_YEAR = 2025


def safe_read_csv(
    url: str,
    retries: int = 3,
    wait_seconds: int = 2,
    request_timeout: int = 20,
    encoding: str = "utf-8",
) -> pd.DataFrame:
    """Read CSV URL with retry logic for rate limits and transient network issues."""
    for attempt in range(retries):
        try:
            with urlopen(url, timeout=request_timeout) as response:
                content = response.read().decode(encoding, errors="replace")
            return pd.read_csv(StringIO(content))
        except HTTPError as err:
            # Retry only for HTTP 429 (rate limit), otherwise treat as no data.
            if err.code == 429 and attempt < retries - 1:
                time.sleep(wait_seconds * (attempt + 1))
                continue
            return pd.DataFrame()
        except (URLError, TimeoutError, socket.timeout):
            if attempt < retries - 1:
                time.sleep(wait_seconds * (attempt + 1))
                continue
            return pd.DataFrame()


def session_rainfall_flag(weather: pd.DataFrame):
    """Return True if any weather sample indicates rainfall; otherwise False/NA."""
    if weather.empty or "rainfall" not in weather.columns:
        return pd.NA

    rainfall_values = weather["rainfall"].dropna()
    if rainfall_values.empty:
        return pd.NA

    # Handle numeric encodings (0/1) and textual/boolean encodings (true/false).
    rainfall_num = pd.to_numeric(rainfall_values, errors="coerce")
    if not rainfall_num.isna().all():
        return bool((rainfall_num > 0).any())

    rainfall_text = rainfall_values.astype(str).str.strip().str.lower()
    return bool(rainfall_text.isin(["1", "true", "yes", "y", "t"]).any())


def session_safety_car_flag(race_control: pd.DataFrame):
    """Return 1 if any race-control row has category SafetyCar, else 0/NA."""
    if race_control.empty or "category" not in race_control.columns:
        return pd.NA

    categories = race_control["category"].dropna().astype(str).str.strip().str.lower()
    if categories.empty:
        return pd.NA

    return int(categories.eq("safetycar").any())


# -----------------------------
# 2. Load race sessions for all requested years
# -----------------------------
session_maps = []

for year in range(START_YEAR, END_YEAR + 1):
    sessions_url = f"https://api.openf1.org/v1/sessions?year={year}&session_type=Race&csv=true"
    sessions = safe_read_csv(sessions_url)

    if sessions.empty:
        continue

    year_map = sessions[["meeting_key", "session_key"]].dropna().drop_duplicates()
    session_maps.append(year_map)

if not session_maps:
    raise ValueError(f"No race sessions found from {START_YEAR} to {END_YEAR}.")

session_map = pd.concat(session_maps, ignore_index=True).drop_duplicates()

# -----------------------------
# 3. Load and merge per-meeting data
# -----------------------------
all_merged = []

for row in session_map.itertuples(index=False):
    meeting_key = int(row.meeting_key)
    race_session_key = int(row.session_key)

    # Final classified result for the race session
    results_url = f"https://api.openf1.org/v1/session_result?session_key={race_session_key}&csv=true"
    results = safe_read_csv(results_url)

    # Starting grid is tied to the meeting
    grid_url = f"https://api.openf1.org/v1/starting_grid?meeting_key={meeting_key}&csv=true"
    grid = safe_read_csv(grid_url)

    if results.empty or grid.empty:
        continue

    # Driver metadata for acronym and team name
    drivers_url = f"https://api.openf1.org/v1/drivers?session_key={race_session_key}&csv=true"
    drivers = safe_read_csv(drivers_url)

    # Session weather to derive a single rainfall flag for the race
    weather_url = f"https://api.openf1.org/v1/weather?session_key={race_session_key}&csv=true"
    weather = safe_read_csv(weather_url)
    rainfall = session_rainfall_flag(weather)

    # Laps to derive each driver's speed-trap metric (st_speed)
    laps_url = f"https://api.openf1.org/v1/laps?session_key={race_session_key}&csv=true"
    laps = safe_read_csv(laps_url)

    # Pit stops to derive each driver's lane duration
    pit_url = f"https://api.openf1.org/v1/pit?session_key={race_session_key}&csv=true"
    pit = safe_read_csv(pit_url)

    # Race control messages to derive a SafetyCar one-hot encoded indicator
    race_control_url = f"https://api.openf1.org/v1/race_control?session_key={race_session_key}&csv=true"
    race_control = safe_read_csv(race_control_url)
    safety_car = session_safety_car_flag(race_control)

    # Stints to derive each driver's starting compound
    stints_url = f"https://api.openf1.org/v1/stints?session_key={race_session_key}&csv=true"
    stints = safe_read_csv(stints_url)

    # Meeting metadata for circuit details
    meetings_url = f"https://api.openf1.org/v1/meetings?meeting_key={meeting_key}&csv=true"
    meetings = safe_read_csv(meetings_url)
    circuit_short_name = pd.NA
    circuit_type = pd.NA
    if not meetings.empty:
        if "circuit_short_name" in meetings.columns:
            non_null_short = meetings["circuit_short_name"].dropna()
            if not non_null_short.empty:
                circuit_short_name = non_null_short.iloc[0]
        if "circuit_type" in meetings.columns:
            non_null_type = meetings["circuit_type"].dropna()
            if not non_null_type.empty:
                circuit_type = non_null_type.iloc[0]

    # Keep only relevant columns from race results
    result_cols = ["session_key", "meeting_key", "driver_number", "position", "position_start"]
    available_result_cols = [col for col in result_cols if col in results.columns]
    results = results[available_result_cols].rename(
        columns={
            "position": "finish_position",
            "position_start": "championship_position_start",
        }
    )
    if "championship_position_start" not in results.columns:
        results["championship_position_start"] = pd.NA

    grid = grid[["meeting_key", "driver_number", "position"]].rename(
        columns={"position": "start_position"}
    )

    merged_session = pd.merge(
        grid,
        results,
        on=["meeting_key", "driver_number"],
        how="inner",
    )

    if not drivers.empty and {"driver_number", "name_acronym", "team_name"}.issubset(drivers.columns):
        drivers = (
            drivers[["driver_number", "name_acronym", "team_name"]]
            .dropna(subset=["driver_number"])
            .drop_duplicates(subset=["driver_number"])
            .rename(columns={"name_acronym": "driver_acronym"})
        )
        merged_session = pd.merge(merged_session, drivers, on="driver_number", how="left")

    if not stints.empty and {"driver_number", "compound"}.issubset(stints.columns):
        available_sort_cols = [col for col in ["lap_start", "stint_number"] if col in stints.columns]
        stints_for_merge = stints[["driver_number", "compound", *available_sort_cols]].dropna(
            subset=["driver_number"]
        )
        if available_sort_cols:
            stints_for_merge = stints_for_merge.sort_values(available_sort_cols)
        stints_for_merge = stints_for_merge.drop_duplicates(subset=["driver_number"], keep="first")
        stints_for_merge = stints_for_merge[["driver_number", "compound"]]
        merged_session = pd.merge(merged_session, stints_for_merge, on="driver_number", how="left")

    if not laps.empty and {"driver_number", "st_speed"}.issubset(laps.columns):
        st_speed_for_merge = (
            laps[["driver_number", "st_speed"]]
            .dropna(subset=["driver_number"]) 
            .assign(st_speed=lambda d: pd.to_numeric(d["st_speed"], errors="coerce"))
            .groupby("driver_number", as_index=False)["st_speed"]
            .max()
        )
        merged_session = pd.merge(merged_session, st_speed_for_merge, on="driver_number", how="left")

    if not pit.empty and {"driver_number", "lane_duration"}.issubset(pit.columns):
        lane_for_merge = (
            pit[["driver_number", "lane_duration"]]
            .dropna(subset=["driver_number"]) 
            .assign(lane_duration=lambda d: pd.to_numeric(d["lane_duration"], errors="coerce"))
            .groupby("driver_number", as_index=False)["lane_duration"]
            .sum(min_count=1)
        )
        merged_session = pd.merge(merged_session, lane_for_merge, on="driver_number", how="left")

    merged_session["rainfall"] = rainfall
    merged_session["safety_car"] = safety_car
    merged_session["circuit_short_name"] = circuit_short_name
    merged_session["circuit_type"] = circuit_type

    # Keep the same final shape plus requested extra fields
    for col in ["driver_acronym", "team_name", "compound", "st_speed", "lane_duration"]:
        if col not in merged_session.columns:
            merged_session[col] = pd.NA

    merged_session = merged_session[
        [
            "session_key",
            "driver_number",
            "driver_acronym",
            "team_name",
            "circuit_short_name",
            "circuit_type",
            "championship_position_start",
            "start_position",
            "finish_position",
            "rainfall",
            "safety_car",
            "compound",
            "st_speed",
            "lane_duration",
        ]
    ]

    all_merged.append(merged_session)

if not all_merged:
    raise ValueError(
        "No merged data could be created. This can happen temporarily when API rate limits are exceeded (429)."
    )

# -----------------------------
# 4. Combine all races and save
# -----------------------------
merged = pd.concat(all_merged, ignore_index=True)
merged.to_csv("f1_start_vs_finish.csv", index=False)

print("Dataset created: f1_start_vs_finish.csv")
print(f"Years included: {START_YEAR}-{END_YEAR}")
print(merged.head())

Dataset created: f1_start_vs_finish.csv
Years included: 2023-2025
   session_key  driver_number driver_acronym        team_name  \
0         7953              1            VER  Red Bull Racing   
1         7953             11            PER  Red Bull Racing   
2         7953             16            LEC          Ferrari   
3         7953             55            SAI          Ferrari   
4         7953             14            ALO     Aston Martin   

  circuit_short_name circuit_type championship_position_start  start_position  \
0             Sakhir    Permanent                         NaN               1   
1             Sakhir    Permanent                         NaN               2   
2             Sakhir    Permanent                         NaN               3   
3             Sakhir    Permanent                         NaN               4   
4             Sakhir    Permanent                         NaN               5   

   finish_position rainfall safety_car compound  st_spee

C:\Users\mterl\AppData\Local\Temp\ipykernel_12408\838260962.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = pd.concat(all_merged, ignore_index=True)
